# 08 · A random-walk prior on the factor dynamics

Notebooks 05–07 kept finding the same thing: the model reverts too fast
towards a historical mean, while the random walk, which does not revert at
all, is hard to beat at 6–12 months. Duffee (2011) finds that the most
useful restriction for forecasting is a random walk for the level, and
Carriero, Kapetanios & Marcellino (2012) shrink yield VARs towards
random-walk-like models. This notebook tests that idea (README §8 step 7b,
item 5).

**The prior** (`yck.model.statespace_dns.rw_prior`, fixed before the run,
commit `04a8463`). A Minnesota prior on the whole state VAR matrix A,
centred on the identity: every state a random walk.

- **Own lags:** N(1, τ²).
- **Cross lags:** N(0, (0.5 τ s_i/s_j)²), with s the two-step shock sds.
- **No prior on Q.**
- **Estimation:** maximised together with the likelihood (MAP).
  τ ∈ {0.03, 0.1, 0.3}: smaller τ gives forecasts closer to a random walk,
  larger τ the plain MLE. μ stays at the real-time sample mean; with A = I
  it drops out of the forecast.
- **Models:** yields only and yields + policy rate (λ fixed), each plain
  and with AR(1) errors.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from yck.eval.backtest import dns_kalman_forecasts, dns_two_step_forecasts, random_walk_forecasts, forward_forecasts
from yck.eval.grid import (STANDARD_GRID, BACKTEST_DEFAULTS, accuracy_grid, anchor_grid, shadow_grid, rwprior_grid,
                           plan_runs,
                           combination_forecasts, ensure_results, tag_of, _grid_for)
from yck.eval.stats import dm_table, mcs_table

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS, CONFIG, DATA = ROOT / "data/results", ROOT / "config/data_sources.yaml", ROOT / "data"
CCYS = ["USD", "EUR", "GBP"]
H = (1, 6, 12)
SHOW = {"USD": ["3m", "12m", "24m", "60m", "120m"], "EUR": ["3m", "12m", "24m", "60m", "120m"],
        "GBP": ["6m", "12m", "24m", "60m", "120m"]}
FIRST_ORIGIN = {c: BACKTEST_DEFAULTS[c][1] for c in CCYS}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CAP = [{"selector": "caption", "props": "font-weight: bold; text-align: left"}]

GRIDS = (STANDARD_GRID, accuracy_grid, anchor_grid, shadow_grid, rwprior_grid)
for g in GRIDS:  # one grid run if anything is missing
    ensure_results(CCYS, DATA, CONFIG, grid=g)

def name_of(row):
    m, lam, tau, ep, err, anc, sr, rw, *_ = tuple(row) + (None,) * (9 - len(row))
    base = "shadow" if sr else ("policy rate" if m == ("policy_rate",) else "full macro" if m else "yields")
    base += "" if lam == "fixed" else ", λ est."
    base += f", shrink {tau:g}" if tau is not None else ""
    base += f", RW prior {rw:g}" if rw is not None else ""
    base += f", endpoint {ep}" if ep else ""
    base += " + AR(1)" if err == "ar1" else " + AR(1) with mean" if err == "ar1c" else ""
    return base + (", anchored" if anc else "") + (f" ({sr})" if sr else "")

y, fc = {}, {}
for c in CCYS:
    est = BACKTEST_DEFAULTS[c][0]
    full = pd.read_parquet(ROOT / f"data/processed/curves_{c}.parquet")
    y[c] = full.loc[est:] if est else full
    fc[c] = {}
    for g in GRIDS:
        for r in _grid_for(g, c):
            fc[c][name_of(r)] = {h: pd.read_parquet(RESULTS / f"fc_{c}_kalman_{tag_of(r)}_h{h}.parquet") for h in H}
    fc[c]["two-step AR(1)"] = dns_two_step_forecasts(y[c], H, FIRST_ORIGIN[c])
    fc[c]["two-step VAR(1)"] = dns_two_step_forecasts(y[c], H, FIRST_ORIGIN[c], kind="var1")
    fc[c]["random walk"] = random_walk_forecasts(y[c], H, FIRST_ORIGIN[c])
    fc[c]["forwards"] = forward_forecasts(pd.read_parquet(ROOT / f"data/processed/curves_ext_{c}.parquet"),
                                          y[c].columns, H, FIRST_ORIGIN[c])
    fc[c].update(combination_forecasts(c, DATA, CONFIG))
PRIOR = [name_of(r) for r in _grid_for(rwprior_grid, "USD")]
BASE_OF = {p: p.split(", RW prior")[0] + (" + AR(1)" if "AR(1)" in p else "") for p in PRIOR}
pd.DataFrame({c: {"models compared": len(fc[c])} for c in CCYS})

,USD,EUR,GBP
models compared,60,54,48


## 1. What the prior does to A

The largest eigenvalue of A and the largest distance |A − I| across the
refits of each model. With τ = 0.03 every state is held close to a random
walk. With τ = 0.3 the estimates are close to the unrestricted ones.

In [2]:
rows = {}
for c in CCYS:
    for m, mtag, k in [("yields", "", 3), ("policy rate", "macro-policy_rate_", 4)]:
        for t in (None,) + (0.03, 0.1, 0.3):
            d = pd.read_csv(RESULTS / f"params_{c}_kalman_{mtag}{'' if t is None else f'rw{t:g}_'}lamfixed.csv")
            A = d[[f"A{i}{j}" for i in range(k) for j in range(k)]].to_numpy().reshape(-1, k, k)
            rad = np.abs(np.linalg.eigvals(A)).max(axis=1)
            rows[(c, m, "none" if t is None else f"τ = {t}")] = {
                "max |eig A|, range over refits": f"{rad.min():.3f}–{rad.max():.4f}",
                "max |A − I|, mean over refits": round(float(np.abs(A - np.eye(k)).max(axis=(1, 2)).mean()), 2)}
pd.DataFrame(rows).T

max |eig A|, range over refits  \
USD yields      none                       0.974–0.9964   
                τ = 0.03                   0.984–0.9945   
                τ = 0.1                    0.976–0.9959   
                τ = 0.3                    0.975–0.9963   
    policy rate none                       0.985–0.9967   
                τ = 0.03                   0.990–0.9982   
                τ = 0.1                    0.980–0.9969   
                τ = 0.3                    0.984–0.9967   
EUR yields      none                       0.953–0.9863   
                τ = 0.03                   0.972–0.9878   
                τ = 0.1                    0.944–0.9879   
                τ = 0.3                    0.951–0.9866   
    policy rate none                       0.949–0.9913   
                τ = 0.03                   0.979–0.9913   
                τ = 0.1                    0.949–0.9922   
                τ = 0.3                    0.946–0.9915   
GBP yields      none                       0.987–0.9963   
                τ = 0.03                   0.986–0.9957   
                τ = 0.1                    0.987–0.9962   
                τ = 0.3                    0.987–0.9963   
    policy rate none                       0.968–0.9973   
                τ = 0.03                   0.993–0.9984   
                τ = 0.1                    0.989–0.9976   
                τ = 0.3                    0.970–0.9972   

                         max |A − I|, mean over refits  
USD yields      none                              0.09  
                τ = 0.03                          0.04  
                τ = 0.1                           0.08  
                τ = 0.3                           0.09  
    policy rate none                              0.35  
                τ = 0.03                          0.05  
                τ = 0.1                            0.1  
                τ = 0.3                           0.25  
EUR yields      none                               0.2  
                τ = 0.03                          0.04  
                τ = 0.1                           0.12  
                τ = 0.3                           0.18  
    policy rate none                              0.56  
                τ = 0.03                          0.04  
                τ = 0.1                           0.12  
                τ = 0.3                           0.32  
GBP yields      none                              0.06  
                τ = 0.03                          0.03  
                τ = 0.1                           0.05  
                τ = 0.3                           0.06  
    policy rate none                              0.35  
                τ = 0.03                          0.04  
                τ = 0.1                           0.08  
                τ = 0.3                           0.27

## 2. Against the same model without the prior

RMSE ratio, prior ÷ no prior, same error model (Diebold–Mariano, two-sided
stars; < 1 means the prior helps).

In [3]:
def after(c, h):
    return y[c].loc[pd.Timestamp(FIRST_ORIGIN[c]) + pd.offsets.MonthEnd(h):]

def vs(c, bench, h, names, field="cell"):
    t = dm_table({n: fc[c][n][h] for n in [bench] + names}, bench, after(c, h), h, SHOW[c])
    return t.xs(field, axis=1, level=1).T

for c in CCYS:
    t = pd.concat({f"h = {h}": pd.DataFrame({p: vs(c, BASE_OF[p], h, [p]).loc[p] for p in PRIOR}).T for h in H},
                  axis=1)
    display(t.style.set_caption(f"{c}: RW prior ÷ no prior").set_table_styles(CAP))

## 3. Against forwards and the random walk

In [4]:
SHOWN = ["yields", "policy rate"] + PRIOR + ["combination A, inverse MSE"]
for c in CCYS:
    t = pd.concat({**{f"÷ forwards, h = {h}": vs(c, "forwards", h, SHOWN) for h in (6, 12)},
                   **{f"÷ RW, h = {h}": vs(c, "random walk", h, SHOWN) for h in (6, 12)}}, axis=1)
    display(t.style.set_caption(f"{c}").set_table_styles(CAP))

## 4. Model confidence set over everything

The 90% MCS over every model in notebooks 02–08 (pooled squared-error
loss), with the p-values of the prior variants, the plain models and the
benchmarks.

In [5]:
SHORT = "3m / 6m (GBP)"
mcs = {(c, h): mcs_table({n: f[h] for n, f in fc[c].items()}, after(c, h), h, SHOW[c], n_boot=2000)
               .rename(columns={SHOW[c][0]: SHORT})
       for c in CCYS for h in H}
summary = {(c, f"h = {h}"): {col: f"{int((t[col] >= 0.10).sum())} / {int(t[col].notna().sum())}" for col in t.columns}
           for (c, h), t in mcs.items()}
display(pd.DataFrame(summary).T.style.set_caption("Models in the 90% MCS / models compared").set_table_styles(CAP))
KEY = ["yields", "policy rate"] + PRIOR + ["combination A, inverse MSE", "random walk", "forwards"]
pd.DataFrame({(c, f"h = {h}"): mcs[(c, h)].loc[KEY, "all"] for c in CCYS for h in H}).round(2)

USD                EUR               \
                                   h = 1 h = 6 h = 12 h = 1 h = 6 h = 12   
model                                                                      
yields                              0.00  0.94   0.99  0.00  0.36   0.55   
policy rate                         0.00  0.98   0.99  0.00  0.34   0.17   
yields, RW prior 0.03               0.00  0.98   0.99  0.00  0.49   0.63   
yields, RW prior 0.03 + AR(1)       0.67  0.98   0.99  0.13  0.49   0.63   
yields, RW prior 0.1                0.00  0.98   0.99  0.00  0.40   0.63   
yields, RW prior 0.1 + AR(1)        0.93  0.98   0.99  0.13  0.34   0.63   
yields, RW prior 0.3                0.00  0.95   0.99  0.00  0.35   0.58   
yields, RW prior 0.3 + AR(1)        0.81  0.98   0.99  0.09  0.31   0.53   
policy rate, RW prior 0.03          0.00  0.71   0.83  0.00  0.49   0.63   
policy rate, RW prior 0.03 + AR(1)  0.41  0.71   0.83  0.13  0.49   0.63   
policy rate, RW prior 0.1           0.00  0.71   0.90  0.00  0.11   0.36   
policy rate, RW prior 0.1 + AR(1)   0.67  0.71   0.91  0.02  0.09   0.33   
policy rate, RW prior 0.3           0.00  0.98   0.99  0.00  0.09   0.14   
policy rate, RW prior 0.3 + AR(1)   0.93  0.98   0.99  0.02  0.06   0.13   
combination A, inverse MSE          1.00  0.98   0.99  0.87  0.96   0.86   
random walk                         0.57  0.98   0.99  0.87  0.96   0.95   
forwards                            1.00  0.98   0.99  1.00  1.00   1.00   

                                     GBP               
                                   h = 1 h = 6 h = 12  
model                                                  
yields                              0.00  0.73   0.86  
policy rate                         0.00  0.73   0.80  
yields, RW prior 0.03               0.00  0.80   0.99  
yields, RW prior 0.03 + AR(1)       0.79  0.80   0.98  
yields, RW prior 0.1                0.00  0.73   0.86  
yields, RW prior 0.1 + AR(1)        0.79  0.73   0.86  
yields, RW prior 0.3                0.00  0.73   0.86  
yields, RW prior 0.3 + AR(1)        0.79  0.73   0.86  
policy rate, RW prior 0.03          0.00  0.33   0.68  
policy rate, RW prior 0.03 + AR(1)  0.79  0.36   0.67  
policy rate, RW prior 0.1           0.00  0.68   0.72  
policy rate, RW prior 0.1 + AR(1)   0.79  0.73   0.70  
policy rate, RW prior 0.3           0.00  0.73   0.86  
policy rate, RW prior 0.3 + AR(1)   0.79  0.73   0.86  
combination A, inverse MSE          0.79  0.97   0.99  
random walk                         0.80  0.98   0.99  
forwards                             NaN  0.90   0.86

## 5. Takeaways

1. **Only the tightest prior matters.** At τ = 0.1 and 0.3 the forecasts
   are within a few percent of the unrestricted model. At τ = 0.03 every
   refit's A stays within 0.03–0.05 of the identity.
2. **EUR gains at every maturity.** With τ = 0.03, RMSE falls 7–16%
   against the same model without the prior at h = 6–12, significant at
   5–10% at most maturities. This is the model's mean-reversion problem
   (README §7.6, failure 2) attacked from the dynamics side.
   - The 10y at h = 6 is now 0.94–0.96 × forwards (not significant).
   - The short end still loses to forwards (1.31–1.59) and to the random
     walk.
3. **GBP gains at the long end.** Yields only, τ = 0.03, at 5–10y:

   | Horizon | Ratio to no prior | Significance |
   |---|---|---|
   | h = 6 | 0.94–0.96 | 5–10% |
   | h = 12 | 0.90–0.93 | not significant |

   The 10y is level with forwards (0.97–1.01) but still 4–8% behind the
   random walk.
4. **USD barely changes.**
   - Yields only, τ = 0.03: 5–10y gain 4–5% (not significant). The 10y at
     h = 12 is 0.93 × forwards, the same as combination A. The short end
     gets 2–5% worse.
   - Policy-rate model: the prior hurts the short end (3m at h = 6: 1.18×,
     10%). It pulls the policy-rate → curve links towards zero, and those
     links were the useful part of that model.
5. **Confidence set.** Still no separation at h = 6–12: it keeps 46–60 of
   48–60 models at every maturity. Policy-rate models with weak priors
   (τ = 0.1–0.3) drop out in EUR at h = 6 (p = 0.06–0.11).
6. **What carries forward.**
   - **Yields only with the τ = 0.03 prior** is the long-end model of
     choice for EUR and GBP. It removes part of the mean-reversion bias
     without a survey or an extra state.
   - **The policy-rate model keeps no prior.**
   - **Combination A is unchanged.**